In [1]:
from pyspark.sql.functions import row_number
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    lag, col, lit, radians, sin, 
    cos, asin, sqrt, unix_timestamp, abs, min
)
from pyspark.sql import functions as F
from pyspark.sql.column import Column
from pyspark.sql.window import Window
from haversine import haversine, Unit
from dataclasses import dataclass
from pyspark import SparkContext
from datetime import datetime
from pyspark.sql.types import *
import numpy as np
import pandas as pd
import os

In [2]:
@dataclass
class Coordinates:
    latitude: np.float64
    longitude: np.float64
    
    @property
    def point(self) -> tuple:
        return (
            self.latitude,
            self.longitude
        )

In [3]:
SOG_MOVE = 1
# 50-nautical-mile
NAUTICAL_MILE = 50
COLLISSION_METERS = 10
COLLISION_TIME_WINDOW_MINUTES = 10


# later
MIN_SECONDS_COLLISION = 60

<h3>Big data analytics Task 4</h3>

In [4]:
data_source_folder = "aisdk-2021-12"
paths = [
    os.path.join(
        data_source_folder, 
        data_source_folder + "-" + str(i).rjust(2, "0") + ".csv"
    )
    for i in range(1, 32)
]

print("Using file names")
print(paths[:3])
print("....")

Using file names
['aisdk-2021-12/aisdk-2021-12-01.csv', 'aisdk-2021-12/aisdk-2021-12-02.csv', 'aisdk-2021-12/aisdk-2021-12-03.csv']
....


In [5]:
spark = SparkSession.builder.appName("task_4_cluster").getOrCreate()
if spark:
    print(f"Task_4 cluster working. Version = {spark.version}")
else:
    print("ERROR: something went wrong")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/06 02:43:07 WARN Utils: Your hostname, homedev-25p, resolves to a loopback address: 127.0.1.1; using 192.168.0.104 instead (on interface wlp0s20f3)
26/06/06 02:43:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 02:43:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Task_4 cluster working. Version = 4.1.2


<h3>Data cleaning and pre-processing</h3>

In [6]:
# spark.stop()

start_time = datetime.now()
start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
print(f"Reading data to Apache Spark start time: {start_time_str}")


data = spark.read.csv(
    paths,
    sep=",",
    header=True,
    inferSchema=True
)

df_filtered = (
    data
    .select("# Timestamp", "MMSI", "Latitude", "Longitude")
    .filter(
        F.col("# Timestamp").isNotNull() &
        F.col("MMSI").isNotNull() &
        F.col("Latitude").isNotNull() &
        F.col("Longitude").isNotNull()
    )
)

end_time = datetime.now()
end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")

time_diff_seconds = (end_time - start_time).seconds

print("----------------------")
print(f"Total {time_diff_seconds} seconds")
print(f"Execution end-time: {end_time_str}")


Reading data to Apache Spark start time: 2026-06-06 02:43:15


----------------------
Total 223 seconds
Execution end-time: 2026-06-06 02:46:59


In [7]:
read_record_count = df_filtered.count()
print(f"Read records {read_record_count} from files .csv")

Read records 318325485 from files .csv


In [8]:
data_clean = (
    data
    .withColumn(
        "timestamp",
        F.to_timestamp(
            F.col("# Timestamp"),
            "dd/MM/yyyy HH:mm:ss"
        )
    )
    .select(
        "timestamp",
        "MMSI",
        "Latitude",
        "Longitude"
    )
)

In [9]:
w = Window.partitionBy("MMSI").orderBy("timestamp")

In [10]:
data_clean = data_clean.withColumn("rn", row_number().over(w))

In [11]:
# filter out non stationary vessels
df_filtered = data_clean.filter(
    col("SOG") > SOG_MOVE
)

In [ ]:
moving_ships = df_filtered.count()
print(f"Selected moving ships: {moving_ships}")

In [13]:
COORDINATE_CENTER = Coordinates(
    latitude=55.225000,
    longitude=14.245000
)
print(f"Analyzing ships near: {COORDINATE_CENTER}")

Analyzing ships near: Coordinates(latitude=55.225, longitude=14.245)


In [14]:
def haversine_nm_value(
    a_lat_col: str, 
    a_lon_col: str, 
    b_lat_value: np.float64, 
    b_lon_value: np.float64
):
    return (
        3440.065 * 2 * asin(
            sqrt(
                sin((radians(a_lat_col) - radians(lit(b_lat_value))) / 2) ** 2 +
                cos(radians(lit(b_lat_value))) *
                cos(radians(a_lat_col)) *
                sin((radians(a_lon_col) - radians(lit(b_lon_value))) / 2) ** 2
            )
        )
    )
    
def haversine_meters(
    a_lat_col: Column,
    a_lon_col: Column,
    b_lat_col: Column,
    b_lon_col: Column
) -> Column:
    return (
        6371000 * 2 * asin(
            sqrt(
                sin((radians(a_lat_col) - radians(b_lat_col)) / 2) ** 2 +
                cos(radians(b_lat_col)) *
                cos(radians(a_lat_col)) *
                sin((radians(a_lon_col) - radians(b_lon_col)) / 2) ** 2
            )
        )
    )

In [15]:
# filter out 50 neutilon miles
df_filtered = data_clean.filter(
    haversine_nm_value(
        "Latitude", "Longitude",
        COORDINATE_CENTER.latitude, 
        COORDINATE_CENTER.longitude
    ) <= NAUTICAL_MILE
)

In [16]:
shiP_records = df_filtered.count()
print(f"Found ships records in radius: {shiP_records}")

Found ships records in radius: 27585630


<h3>Simple data exploration</h3>

<h3>Collision analysis</h3>

In [17]:
GRID = 0.01  # about 1 km

df_filtered = (
    df_filtered
    .withColumn("lat_bucket", F.floor(F.col("Latitude") / GRID))
    .withColumn("lon_bucket", F.floor(F.col("Longitude") / GRID))
    .withColumn(
        "time_bucket",
        F.floor(
            F.unix_timestamp("timestamp") / 60
        )
    )
)

In [20]:
df_filtered_a = df_filtered.select(
    "timestamp",
    "MMSI",
    "Latitude",
    "Longitude",
    "lat_bucket",
    "lon_bucket",
    "time_bucket"
).alias("a")

df_filtered_b = df_filtered.select(
    "timestamp",
    "MMSI",
    "Latitude",
    "Longitude",
    "lat_bucket",
    "lon_bucket",
    "time_bucket"
).alias("b")

In [21]:
candidate_pairs = (
    df_filtered_a
    .join(
        df_filtered_b,
        (col("a.MMSI") < col("b.MMSI"))
        &
        (col("a.time_bucket") == col("b.time_bucket"))
        &
        (col("a.lat_bucket") == col("b.lat_bucket"))
        &
        (col("a.lon_bucket") == col("b.lon_bucket"))
    )
    .select(
        col("a.MMSI").alias("mmsi_a"),
        col("a.timestamp").alias("timestamp_a"),
        col("a.Latitude").alias("lat_a"),
        col("a.Longitude").alias("lon_a"),

        col("b.MMSI").alias("mmsi_b"),
        col("b.timestamp").alias("timestamp_b"),
        col("b.Latitude").alias("lat_b"),
        col("b.Longitude").alias("lon_b"),
    )
)

In [22]:
dist_df = candidate_pairs.withColumn(
    "distance_m",
    haversine_meters(
        col("lat_a"), col("lon_a"),
        col("lat_b"), col("lon_b")
    )
)

min_dist = dist_df.agg(F.min("distance_m")).first()[0]

closest_pair = (
    dist_df
    .filter(F.col("distance_m") == min_dist)
    .first()
)

In [23]:
closest_pair_dict = closest_pair.asDict()
print(f"Ship pair, which is close to collision: {closest_pair_dict}")

Ship pair, which is close to collision: {'mmsi_a': 265764760, 'timestamp_a': datetime.datetime(2021, 12, 12, 18, 48, 5), 'lat_a': 55.58722, 'lon_a': 12.924328, 'mmsi_b': 265771450, 'timestamp_b': datetime.datetime(2021, 12, 12, 18, 48, 34), 'lat_b': 55.58722, 'lon_b': 12.924328, 'distance_m': 0.0}


In [ ]:
path_a = df_filtered.filter(
    (F.col("MMSI") == closest_pair_dict["mmsi_a"])
    &
    (
        F.abs(
            F.unix_timestamp("timestamp")
            - F.unix_timestamp(F.lit(closest_pair_dict["timestamp_a"]))
        ) <= COLLISION_TIME_WINDOW_MINUTES * 60 
    )
)
path_a.show(truncate=False)

+-------------------+---------+--------+---------+----+----------+----------+-----------+
|timestamp          |MMSI     |Latitude|Longitude|rn  |lat_bucket|lon_bucket|time_bucket|
+-------------------+---------+--------+---------+----+----------+----------+-----------+
|2021-12-12 18:38:08|265764760|55.58722|12.924328|8606|5558      |1292      |27322118   |
|2021-12-12 18:39:10|265764760|55.58722|12.924328|8607|5558      |1292      |27322119   |
|2021-12-12 18:42:09|265764760|55.58722|12.924328|8608|5558      |1292      |27322122   |
|2021-12-12 18:44:11|265764760|55.58722|12.924328|8609|5558      |1292      |27322124   |
|2021-12-12 18:44:11|265764760|55.58722|12.924328|8610|5558      |1292      |27322124   |
|2021-12-12 18:45:07|265764760|55.58722|12.924328|8611|5558      |1292      |27322125   |
|2021-12-12 18:48:05|265764760|55.58722|12.924328|8612|5558      |1292      |27322128   |
|2021-12-12 18:50:09|265764760|55.58722|12.924328|8613|5558      |1292      |27322130   |
|2021-12-1

In [25]:
path_b = df_filtered.filter(
    (F.col("MMSI") == closest_pair_dict["mmsi_b"])
    &
    (
        F.abs(
            F.unix_timestamp("timestamp")
            - F.unix_timestamp(F.lit(closest_pair_dict["timestamp_b"]))
        ) <= COLLISION_TIME_WINDOW_MINUTES * 60 
    )
)
path_b.show(10, truncate=False)

+-------------------+---------+---------+---------+-----+----------+----------+-----------+
|timestamp          |MMSI     |Latitude |Longitude|rn   |lat_bucket|lon_bucket|time_bucket|
+-------------------+---------+---------+---------+-----+----------+----------+-----------+
|2021-12-12 18:38:42|265771450|55.587197|12.924323|10108|5558      |1292      |27322118   |
|2021-12-12 18:42:44|265771450|55.587197|12.924323|10109|5558      |1292      |27322122   |
|2021-12-12 18:44:43|265771450|55.58722 |12.924328|10110|5558      |1292      |27322124   |
|2021-12-12 18:48:34|265771450|55.58722 |12.924328|10111|5558      |1292      |27322128   |
|2021-12-12 18:48:47|265771450|55.58722 |12.924328|10112|5558      |1292      |27322128   |
|2021-12-12 18:50:45|265771450|55.587258|12.924313|10113|5558      |1292      |27322130   |
|2021-12-12 18:53:43|265771450|55.587235|12.924322|10114|5558      |1292      |27322133   |
|2021-12-12 18:54:35|265771450|55.587235|12.924322|10115|5558      |1292      |2

<h3>Results export to csv</h3>

In [ ]:
path_a.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("ship_a_collision")

In [27]:
path_b.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("ship_b_collision")

In [ ]:
spark.stop()

In [ ]:
# The objective of this examination is to 
# evaluate your ability to process large-scale temporal and spatial data. 
# You are required to identify two vessels that have collided 
# (or experienced the closest possible physical proximity indicating a collision) 
# within a specified marine area. You must visualize their respective trajectories 
# 10 minutes prior to and 10 minutes following the time of collision.

In [ ]:
# Geographic Area: 
# 1. Filter the dataset to isolate vessels operating 
# within a 50-nautical-mile (nm) radius of a center coordinate 
# located at Latitude: 55.225000, Longitude: 14.245000.

# 2. Vessel State: You are looking specifically for moving vessels 
# that intersect in time and space, resulting in a collision.
# You must implement logic to identify and filter out stationary 
# vessels (e.g., ships at anchor or safely docked adjacent to one another).

# 3. Data Integrity: AIS data frequently contains errors. 
# You must account for and filter out GPS anomalies and data noise. 
# This is critical to ensure that a sudden jump in GPS coordinates 
# is not falsely identified as a collision.
